In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
from pyspark.sql import functions as F

silver_5m = spark.table("workspace.vol_project.silver_btc_5m")

In [0]:
gold = (
    silver_5m
    .withColumn("date", F.to_date("bucket"))
    .groupBy("date")
    .agg(
        F.sum(F.col("log_ret") ** 2).alias("rv"),
        F.sum("log_ret").alias("daily_ret"),
        F.count("log_ret").alias("n_returns"),
        F.max(F.col("is_gap").cast("int")).alias("has_gap"),
        F.sum("volume").alias("volume")
    )
    .withColumn("rv_ann_vol", F.sqrt(F.col("rv") * 365))
    .withColumn("log_rv", F.log("rv"))
    .orderBy("date")
)

display(gold)

In [0]:
print("days:", gold.count())
print("gap days:", gold.filter("has_gap = 1").count())
display(gold.orderBy(F.desc("rv")).limit(10))

In [0]:
import matplotlib.pyplot as plt

pdf = gold.select("date", "rv_ann_vol").toPandas()

plt.figure(figsize=(14, 5))
plt.plot(pdf["date"], pdf["rv_ann_vol"], linewidth=0.7)
plt.title("BTC Daily Realized Volatility (annualized, 5-min returns)")
plt.ylabel("Annualized vol")
plt.show()

In [0]:
gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.vol_project.gold_btc_daily_rv")